# Notebook Overview

This notebook is used to fine-tune a BERT-based named entity recognition (NER) model using user-authored scenarios describing how users interact with a screen in a mobile app. The notebook covers loading, formatting and splitting the data for training, configuration of the algorithm to train the model, and training the model.

In [1]:
"""! pip install transformers datasets tokenizers evaluate
! pip install transformers[sentencepiece]
! pip install torch
! pip install tensorflow
! pip install spacy
! pip install seqeval
! pip install ipywidgets
! pip install "ray[tune]" scipy scikit-learn"""
print("Passed")

Passed


In [2]:
! pip install datasets seqeval evaluate transformers tokenizers torch tensorflow spacy ipywidgets "ray[tune]" scipy scikit-learn

  Using cached tensorflow-2.20.0-cp39-cp39-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached ray-2.51.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (21 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.80.0-cp39-cp39-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0-py3-no

In [3]:
import json, datasets

def create_dataset(data):
    res = {'words': [], 'tags': []}
    for scenario in data:
        res['words'].append([word for sent in scenario['tokens'] for word, pos, tag in sent])
        res['tags'].append([tag for sent in scenario['tokens'] for word, pos, tag in sent])
    
    dataset = datasets.Dataset.from_dict(res, features=datasets.Features({
        "words": datasets.Sequence(datasets.Value("string")),
        "tags": datasets.Sequence(
            datasets.features.ClassLabel(
                names=['O', 'B-SIM', 'I-SIM', 'B-COM', 'I-COM', 'B-QUE', 'I-QUE']
            )
        ),
    }))
    return dataset

source_data = json.load(open('scenarios-training-N.json'))
prepared_data = datasets.dataset_dict.DatasetDict({
    'train': create_dataset(source_data['train']),
    'validation': create_dataset(source_data['validation']),
    'test': create_dataset(source_data['test'])
})

In [4]:
# tokenize and align dataset
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            if label % 2 == 1:
                label += 1
            new_labels.append(label)
    return new_labels

def tokenize_and_align_labels(examples, **kwargs):
    '''
    Input: a row of Dataset
    Output: use dataset.map() method to map this function for each row.
            This will tokenize each row and align the original labels.
    '''
    tokenizer = kwargs['tokenizer']
    tokenized_inputs = tokenizer(
        examples["words"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [5]:
from sklearn.preprocessing import MultiLabelBinarizer
from seqeval.metrics import classification_report

# compute evaluation for precision, recall, f1 and accuracy
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    MultiLabelBinarizer().fit_transform(true_labels)
    MultiLabelBinarizer().fit_transform(true_predictions)
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    
    # return all_metrics
    print(classification_report(true_labels,true_predictions ))
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [6]:
# load the dataset from the given dataset path

ner_feature = prepared_data["train"].features["tags"]
label_names = ner_feature.feature.names

In [7]:
! pip install tqdm

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
import evaluate
from transformers import Trainer, TrainingArguments

# tokenize the data for fine-tuning the bert-base-NER model
model_checkpoint = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenized_datasets = prepared_data.map(
    tokenize_and_align_labels,
    batched = True,
    remove_columns = prepared_data["train"].column_names,
    fn_kwargs = {'tokenizer': tokenizer}
)

# setup collator and evalation metric
data_collator = DataCollatorForTokenClassification(tokenizer = tokenizer)

# define the model initialization
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

def model_init():
    return AutoModelForTokenClassification.from_pretrained(
            model_checkpoint,
            id2label = id2label,
            label2id = label2id,
            ignore_mismatched_sizes = True
        )
    
# train the model
args = TrainingArguments(
    "bert-finetuned-ner",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    num_train_epochs = 4,
    weight_decay = 0.01,
    push_to_hub = False,
)
trainer = Trainer(
    # model=model,
    model_init = model_init,
    args = args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    tokenizer = tokenizer,
)

# set dependencies for the train() function to complete
import numpy as np
import os
os.environ["WANDB_DISABLED"] = "true"
metric = evaluate.load("seqeval")
print("123")
trainer.train()

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import os

# 1. 转化为绝对路径，消除歧义
model_path = os.path.abspath('./bert-finetuned-ner/checkpoint-200')

# 2. 检查一下路径是否存在（防止刚才环境重造时路径丢了）
if not os.path.exists(model_path):
    print(f"❌ 警告：路径不存在 {model_path}，请确认文件夹名是否正确！")
else:
    # 3. 实例化
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path, ignore_mismatched_sizes=True)
    device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
    
    nlp = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy='first', device = torch.device(device))
    print("✅ Pipeline 加载成功！")

In [ ]:
from spacy.training import offsets_to_biluo_tags, biluo_to_iob
import spacy, json

# setup the spaCy English tokenizer
nlp_parser = spacy.load("en_core_web_sm")

# load the scenario data to obtain untokenized text
dataset = json.load(open('../datasets/scenarios-labeled.json', 'r'))

labels = ['O', 'B-SIM', 'I-SIM', 'B-COM', 'I-COM', 'B-QUE', 'I-QUE']
y_true = []
y_pred = []

for scenario, prepared in zip(source_data['test'], prepared_data['test']):
    for tag in prepared['tags']:
        y_true.append(labels[tag])
        
    # predict the named entities from the test scenario
    text = scenario['text']
    entities = nlp(text)
    
    entity_triples = []
    for entity in entities:
        entity_triples.append([entity['start'], entity['end'], entity['entity_group']])

    # convert character-level label spans to BILUO tags
    doc = nlp_parser(text)
    biluo_tags = offsets_to_biluo_tags(doc, entity_triples)
    
    # conver BILUO tags to BIO tags
    y_pred.extend(biluo_to_iob(biluo_tags))
    
print('Created y_true length = %i' % len(y_true))
print('Created y_pred length = %i' % len(y_pred))

/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "My guilty pleasure is using the McDonalds app. I u..." with entities "[[81, 87, 'SIM'], [136, 140, 'SIM'], [177, 186, 'S...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I use this screen as part of the video recording a..." with entities "[[33, 48, 'SIM'], [131, 139, 'SIM'], [243, 249, 'S...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Created y_true length = 8822
Created y_pred length = 8822


/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "I like to use this screen to gauge how much I am b..." with entities "[[35, 56, 'QUE'], [61, 108, 'QUE'], [155, 163, 'SI...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


In [ ]:
from sklearn.metrics import classification_report
    
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           -       0.00      0.00      0.00         0
       B-COM       0.00      0.00      0.00        39
       B-QUE       0.64      0.87      0.74        79
       B-SIM       0.66      0.76      0.71       626
       I-COM       0.60      0.02      0.03       177
       I-QUE       0.82      0.83      0.82       462
       I-SIM       0.51      0.42      0.46       107
           O       0.95      0.96      0.96      7332

    accuracy                           0.91      8822
   macro avg       0.52      0.48      0.47      8822
weighted avg       0.91      0.91      0.90      8822



/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [ ]:
from seqeval.metrics import classification_report

print(classification_report([y_true], [y_pred]))

              precision    recall  f1-score   support

         COM       0.00      0.00      0.00        39
         QUE       0.49      0.66      0.56        79
         SIM       0.64      0.74      0.69       626
           _       0.00      0.00      0.00         0

   micro avg       0.60      0.69      0.64       744
   macro avg       0.28      0.35      0.31       744
weighted avg       0.59      0.69      0.64       744



/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: - seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


  Using cached murmurhash-1.0.15-cp39-cp39-macosx_11_0_arm64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp39-cp39-macosx_11_0_arm64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.13-cp39-cp39-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached wrapt-2.1.2-cp39-cp39-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 30.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 658.8/658.8 kB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 kB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━